# Covariate / samplestokeep builder


## Load libraries

In [ ]:
import os
import sys
import pathlib
from pathlib import Path

from itertools import product
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

import subprocess
import pandas as pd
import numpy as np

from datetime import datetime, date
import time

## Paths 

In [ ]:
release = 12

RELEASE_PATH = pathlib.Path(pathlib.Path.home(), f'workspace/gp2_tier2_eu_release{release}')
PATH_CLINICAL = pathlib.Path(RELEASE_PATH, 'clinical_data/master_key_release12_final_vwb.csv')

# NBA
PATH_NBA_RAW     = pathlib.Path(RELEASE_PATH, 'raw_genotypes')
PATH_NBA_RELATED = pathlib.Path(RELEASE_PATH, 'meta_data/related_samples')

# WGS
PATH_WGS_RAW     = pathlib.Path(RELEASE_PATH, 'raw_genotypes')
PATH_WGS_RELATED = pathlib.Path(RELEASE_PATH, 'wgs/dragen_joint_calling/related_samples')

dataset = ["NBA", "WGS"]
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
print(ANCESTRIES)

## Working directories 

In [ ]:
DIR_TOOL = "/home/jupyter/tools"
DIR_HOME = "/home/jupyter/workspace"
DIR_WSPS = "/home/jupyter/workspace/ws_files"

DIR_NOVA = f"{DIR_WSPS}/Novalis_v3_R12"
MASTER_SAMPLESTOKEEP = f"{DIR_NOVA}/samplestokeep.txt"

DIR_WORK_WGS = f"{DIR_WSPS}/Working_WGS_v3"
DIR_WORK_NBA = f"{DIR_WSPS}/Working_NBA_v3"

dirs = [DIR_TOOL, DIR_HOME, DIR_WSPS, DIR_WORK_WGS, DIR_WORK_NBA]
subdirs = ["InputFiles", "VariantDescriptives", "Association", "GLM", "Burden"]

for d in dirs:
    os.makedirs(d, exist_ok=True)

for ancestry in ANCESTRIES:
    for subdir in subdirs:
        os.makedirs(pathlib.Path(DIR_WORK_WGS, ancestry, subdir), exist_ok=True)
        os.makedirs(pathlib.Path(DIR_WORK_NBA, ancestry, subdir), exist_ok=True)

print("Working directories ready.")
print(f"Master samplestokeep.txt will be written to: {MASTER_SAMPLESTOKEEP}")

## Build the master samplestokeep list


In [ ]:
key_full = pd.read_csv(PATH_CLINICAL, low_memory=False, usecols=['GP2ID', 'GP2_phenotype_for_qc'])
pheno_map = {'PD': 2, 'Control': 1}

master_stk = key_full[key_full['GP2_phenotype_for_qc'].isin(pheno_map)].copy()
master_stk['PHENO_MASTER'] = master_stk['GP2_phenotype_for_qc'].map(pheno_map)
master_stk = master_stk[['GP2ID', 'PHENO_MASTER']]

os.makedirs(DIR_NOVA, exist_ok=True)
master_stk.to_csv(MASTER_SAMPLESTOKEEP, sep='\t', index=False, header=False)

master_ids_keep = set(master_stk['GP2ID'])
n_pd = (master_stk['PHENO_MASTER'] == 2).sum()
n_ctrl = (master_stk['PHENO_MASTER'] == 1).sum()
print(f"Master samplestokeep.txt: {len(master_stk)} individuals ({n_pd} PD, {n_ctrl} control) -> {MASTER_SAMPLESTOKEEP}")

## Load master key and build the ancestry label column

In [ ]:
key1 = pd.read_csv(PATH_CLINICAL, low_memory=False)
print(f'Clinical data (rows, cols): {key1.shape}')

key = key1[['GP2ID', 'GP2ID', 'GP2_phenotype_for_qc', 'biological_sex_for_qc',
            'age_at_sample_collection', 'age_of_onset', 'nba_label', 'wgs_label']].copy()
key.columns = ['IID', 'IID1_OLD', 'phenotype', 'SEX', 'AGE', 'AAO', 'nba_label', 'wgs_label']

key["label"] = key["nba_label"].combine_first(key["wgs_label"])
key = key.drop(columns=["nba_label", "wgs_label"])

print(f'Total individuals in master key: {key.shape[0]}')
key.head()

## Build `{ANCESTRY}.samplestokeep` and `{ANCESTRY}_covariate_file.txt`


In [ ]:
summary_rows = []

for data in dataset:
    for ANCESTRY in ANCESTRIES:

        print(f'\n===== {data} - {ANCESTRY} =====')

        DIR_WORK = DIR_WORK_NBA if data == "NBA" else DIR_WORK_WGS
        PATH_REL = PATH_NBA_RELATED if data == "NBA" else PATH_WGS_RELATED
        PATH_RAW = PATH_NBA_RAW if data == "NBA" else PATH_WGS_RAW

        MAIN = f"{DIR_WORK}/{ANCESTRY}"

        ANCESTRY_key = key[key['label'] == ANCESTRY].copy()
        n_anc = len(ANCESTRY_key)

        ANCESTRY_key = ANCESTRY_key[ANCESTRY_key['IID'].isin(master_ids_keep)].copy()
        n_pdctrl = len(ANCESTRY_key)
        n_excluded_unknown = n_anc - n_pdctrl
        print(f'  {ANCESTRY}: {n_anc} individuals -> {n_pdctrl} after dropping Unknown/Other ({n_excluded_unknown} excluded)')
        ANCESTRY_key = ANCESTRY_key.reset_index(drop=True)

        if n_pdctrl == 0:
            print(f'  {data}-{ANCESTRY}: no PD/Control individuals left, skipping.')
            summary_rows.append(dict(dataset=data, ancestry=ANCESTRY, n_ancestry=n_anc,
                                          n_pd_control=0, n_excluded_unknown=n_excluded_unknown,
                                          n_related_removed=None, n_final=0, status='NO_DATA'))
            continue

        related_df = None
        try:
            if data == "NBA":
                related_df = pd.read_csv(f'{PATH_REL}/{ANCESTRY}_release12_vwb.related')
            elif data == "WGS":
                related_df = pd.read_csv(f'{PATH_REL}/{ANCESTRY}/{ANCESTRY}_release12.related')
        except Exception as e:
            print(f'  Could not read .related file for {data}-{ANCESTRY}: {e}')

        if related_df is None:
            print(f'  {data}-{ANCESTRY}: no .related file, continuing without removing related individuals.')
            related_list = []
        else:
            related_list = list(related_df['IID1'])

        n_before_rel = len(ANCESTRY_key)
        ANCESTRY_key = ANCESTRY_key[~ANCESTRY_key["IID1_OLD"].isin(related_list)]
        n_after_rel = len(ANCESTRY_key)
        print(f'  After removing related individuals: {n_before_rel} -> {n_after_rel}')

        # Phenotype to binary PHENO (1=Control, 2=PD)
        pheno_mapping = {"PD": 2, "Control": 1}
        ANCESTRY_key['PHENO'] = ANCESTRY_key['phenotype'].map(pheno_mapping).astype('Int64')

        # PCs
        pcs_path = f'{PATH_RAW}/{ANCESTRY}/{ANCESTRY}_release12_vwb.eigenvec'
        try:
            pcs = pd.read_csv(pcs_path, sep='\t')
        except Exception as e:
            print(f'  Could not read eigenvec for {data}-{ANCESTRY}: {e}')
            summary_rows.append(dict(dataset=data, ancestry=ANCESTRY, n_ancestry=n_anc,
                                          n_pd_control=n_pdctrl, n_excluded_unknown=n_excluded_unknown,
                                          n_related_removed=n_before_rel - n_after_rel,
                                          n_final=None, status='NO_PCS'))
            continue

        selected_columns = ['FID', 'IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
        pcs = pd.DataFrame(data=pcs.iloc[:, 0:12].values, columns=selected_columns)
        pcs = pcs.drop(0).reset_index(drop=True)

        sex_mapping = {"Female": 2, "Male": 1}
        ANCESTRY_key['SEX'] = ANCESTRY_key['SEX'].map(sex_mapping).astype('Int64')

        # Build covariate file
        df = pd.merge(pcs, ANCESTRY_key, on='IID', how='inner')
        df['FATID'] = 0
        df['MATID'] = 0

        final_df = df[['FID', 'IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO',
                        'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']].copy()

        n_pd_final = (final_df['PHENO'] == 2).sum()
        n_ctrl_final = (final_df['PHENO'] == 1).sum()
        print(f'  Final: {len(final_df)} individuals | PD={n_pd_final} | Control={n_ctrl_final}')

        samples_toKeep = final_df[['FID', 'IID']].copy()

        samplestokeep_path = pathlib.Path(f'{MAIN}/InputFiles/{ANCESTRY}.samplestokeep')
        os.makedirs(samplestokeep_path.parent, exist_ok=True)
        samples_toKeep.to_csv(samplestokeep_path, sep='\t', index=False, header=None)

        finaldf_path = pathlib.Path(f'{MAIN}/InputFiles/{ANCESTRY}_covariate_file.txt')
        os.makedirs(finaldf_path.parent, exist_ok=True)
        final_df.to_csv(finaldf_path, sep='\t', na_rep='NA', index=False)

        summary_rows.append(dict(dataset=data, ancestry=ANCESTRY, n_ancestry=n_anc,
                                      n_pd_control=n_pdctrl, n_excluded_unknown=n_excluded_unknown,
                                      n_related_removed=n_before_rel - n_after_rel,
                                      n_final=len(final_df), status='OK'))

## Summary 

In [ ]:
summary_df = pd.DataFrame(summary_rows)

total_excluded = summary_df['n_excluded_unknown'].sum()
print(f'Total Unknown/Other individuals excluded (summed over NBA+WGS x 11 ancestries): {total_excluded}')
print('(Note: the same person can be counted once in NBA and once in WGS if they have both data types.)')

summary_path = f"{DIR_NOVA}/covariate_builder_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f'Per-ancestry/dataset breakdown saved to: {summary_path}')